# Data Preparation

## Set Up Environment

### Install Dependencies

In [4]:
!pip install --upgrade pandas sentence-transformers pandas tqdm urllib3 Sastrawi

### Import Dependencies

In [90]:
import pandas as pd
import numpy as np
import re
import os

from sentence_transformers import SentenceTransformer
from tqdm import tqdm
from urllib.parse import urlparse
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

In [91]:

from sentence_transformers import SentenceTransformer
from tqdm import tqdm
from urllib.parse import urlparse
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

### Load Model Sentence-BERT (IndoSBERT)

In [92]:
model_name = 'denaya/indoSBERT-large'

model = SentenceTransformer(model_name)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 24563.36it/s]
BertModel LOAD REPORT from: denaya/indoSBERT-large
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### Load Raw Data

In [181]:
data_path = 'datasets/raw_tickets.csv'

df_raw = pd.read_csv(data_path)

df = df_raw[['DESKRIPSI']].dropna()
df.rename(columns={'DESKRIPSI': 'RAW TICKET'}, inplace=True)

df.head()

,RAW TICKET
0,Klien: Setwapres Medsos\nisu: Crawlback Commen...
1,"selamat pagi tim it, mohon bantuannya saya men..."
2,Klien: BPS\nIsu: Data postingan Instagram tida...
3,selamat sore mas @Dhanysybn dan tim info untuk...
4,Klien: Heinz \nIsu: Dashboard loading\n\nSelam...


## Data Preprocessing

### Preprocessed Data for Sentence-Embedding Model

#### Delete Duplicate Data

In [182]:
# hapus data yang duplikat
print(f"Jumlah data sebelum dihapus duplikat: {len(df)}")

# df.drop_duplicates(inplace=True)

print(f"Jumlah data setelah dihapus duplikat: {len(df)}")

Jumlah data sebelum dihapus duplikat: 1639
Jumlah data setelah dihapus duplikat: 1639


#### Replace new line special character

In [183]:
# jika terdapat lebih dari satu newline, ganti dahulu menjadi satu newline, lalu ganti newline tersebut dengan titik dan spasi
df['REPLACED NEWLINE'] = df['RAW TICKET'].apply(lambda x: re.sub(r'\n+', '. ', x))

df.head()

,RAW TICKET,REPLACED NEWLINE
0,Klien: Setwapres Medsos\nisu: Crawlback Commen...,Klien: Setwapres Medsos. isu: Crawlback Commen...
1,"selamat pagi tim it, mohon bantuannya saya men...","selamat pagi tim it, mohon bantuannya saya men..."
2,Klien: BPS\nIsu: Data postingan Instagram tida...,Klien: BPS. Isu: Data postingan Instagram tida...
3,selamat sore mas @Dhanysybn dan tim info untuk...,selamat sore mas @Dhanysybn dan tim info untuk...
4,Klien: Heinz \nIsu: Dashboard loading\n\nSelam...,Klien: Heinz . Isu: Dashboard loading. Selamat...


#### Lower Casing

In [184]:
df['LOWERCASE'] = df['REPLACED NEWLINE'].str.lower()

df.head()

,RAW TICKET,REPLACED NEWLINE,LOWERCASE
0,Klien: Setwapres Medsos\nisu: Crawlback Commen...,Klien: Setwapres Medsos. isu: Crawlback Commen...,klien: setwapres medsos. isu: crawlback commen...
1,"selamat pagi tim it, mohon bantuannya saya men...","selamat pagi tim it, mohon bantuannya saya men...","selamat pagi tim it, mohon bantuannya saya men..."
2,Klien: BPS\nIsu: Data postingan Instagram tida...,Klien: BPS. Isu: Data postingan Instagram tida...,klien: bps. isu: data postingan instagram tida...
3,selamat sore mas @Dhanysybn dan tim info untuk...,selamat sore mas @Dhanysybn dan tim info untuk...,selamat sore mas @dhanysybn dan tim info untuk...
4,Klien: Heinz \nIsu: Dashboard loading\n\nSelam...,Klien: Heinz . Isu: Dashboard loading. Selamat...,klien: heinz . isu: dashboard loading. selamat...


#### Remove Klien and Placeholder 'Isu/Kendala'

In [185]:
def bersihkan_tiket_revisi(teks):
    if not isinstance(teks, str):
        return teks
    
    # 1a. Tiket berformat label: buang blok klien sampai sebelum label isu/kendala
    pattern_klien_berlabel = r'\b(?:(?:klien|client|clien|lien)[\s:;,\.]*|(?:dashboard trial|dashboard client|dashboard|project)\s*[:;]\s*)(.*?)(?=\b(?:isu|issue|kendala|request|req|keluhan|problem|kebutuhan|detail)\b)'
    teks = re.sub(pattern_klien_berlabel, '', teks)
    
    # 1b. Sisa "klien: <nama>" yang mungkin masih ada (tiket tanpa label isu)
    # Hanya buang token "klien:" dan nilai langsung setelahnya sampai titik/koma berikutnya
    pattern_klien_sisa = r'\b(?:klien|client|clien|lien)[\s:;,\.]*[^.!?\n]*?(?=[.!?\n]|$)'
    teks = re.sub(pattern_klien_sisa, '', teks)
    
    # 2. Hapus label isu/kendala (tidak berubah)
    pattern_isu_detail = r'\b(?:isu|issue|kendala|request|req|keluhan|problem|kebutuhan|detail)\s*[:;]\s*'
    teks = re.sub(pattern_isu_detail, '', teks)
    
    # 3. Finalisasi
    teks = teks.strip(' ;:,-')
    teks = re.sub(r'\s+', ' ', teks)
    
    return teks

# Terapkan fungsinya
df['CLEANED TICKET'] = df['LOWERCASE'].apply(bersihkan_tiket_revisi)
df.head()

,RAW TICKET,REPLACED NEWLINE,LOWERCASE,CLEANED TICKET
0,Klien: Setwapres Medsos\nisu: Crawlback Commen...,Klien: Setwapres Medsos. isu: Crawlback Commen...,klien: setwapres medsos. isu: crawlback commen...,crawlback comment. siang tim it minta tolong b...
1,"selamat pagi tim it, mohon bantuannya saya men...","selamat pagi tim it, mohon bantuannya saya men...","selamat pagi tim it, mohon bantuannya saya men...","selamat pagi tim it, mohon bantuannya saya men..."
2,Klien: BPS\nIsu: Data postingan Instagram tida...,Klien: BPS. Isu: Data postingan Instagram tida...,klien: bps. isu: data postingan instagram tida...,data postingan instagram tidak masuk. selamat ...
3,selamat sore mas @Dhanysybn dan tim info untuk...,selamat sore mas @Dhanysybn dan tim info untuk...,selamat sore mas @dhanysybn dan tim info untuk...,selamat sore mas @dhanysybn dan tim info untuk...
4,Klien: Heinz \nIsu: Dashboard loading\n\nSelam...,Klien: Heinz . Isu: Dashboard loading. Selamat...,klien: heinz . isu: dashboard loading. selamat...,"dashboard loading. selamat sore tim it, mohon ..."


#### Masking Nama dan Alamat Email

In [186]:
# Masking nama akun (contoh: @namaakun) dengan placeholder nama orang
df['MASKED TICKET'] = df['CLEANED TICKET'].apply(lambda x: re.sub(r'@\w+', 'nama orang', x))

# Masking email (contoh: email@domain.com) dengan placeholder email
df['MASKED TICKET'] = df['MASKED TICKET'].apply(lambda x: re.sub(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b', 'email@domain.com', x))

df.head()

,RAW TICKET,REPLACED NEWLINE,LOWERCASE,CLEANED TICKET,MASKED TICKET
0,Klien: Setwapres Medsos\nisu: Crawlback Commen...,Klien: Setwapres Medsos. isu: Crawlback Commen...,klien: setwapres medsos. isu: crawlback commen...,crawlback comment. siang tim it minta tolong b...,crawlback comment. siang tim it minta tolong b...
1,"selamat pagi tim it, mohon bantuannya saya men...","selamat pagi tim it, mohon bantuannya saya men...","selamat pagi tim it, mohon bantuannya saya men...","selamat pagi tim it, mohon bantuannya saya men...","selamat pagi tim it, mohon bantuannya saya men..."
2,Klien: BPS\nIsu: Data postingan Instagram tida...,Klien: BPS. Isu: Data postingan Instagram tida...,klien: bps. isu: data postingan instagram tida...,data postingan instagram tidak masuk. selamat ...,data postingan instagram tidak masuk. selamat ...
3,selamat sore mas @Dhanysybn dan tim info untuk...,selamat sore mas @Dhanysybn dan tim info untuk...,selamat sore mas @dhanysybn dan tim info untuk...,selamat sore mas @dhanysybn dan tim info untuk...,selamat sore mas nama orang dan tim info untuk...
4,Klien: Heinz \nIsu: Dashboard loading\n\nSelam...,Klien: Heinz . Isu: Dashboard loading. Selamat...,klien: heinz . isu: dashboard loading. selamat...,"dashboard loading. selamat sore tim it, mohon ...","dashboard loading. selamat sore tim it, mohon ..."


#### Masking Sapaan

In [187]:
# Masking sapaan dengan placeholder sapaan
sapaan_pattern = (
    r'\b('
    # Sapaan berbasis waktu (panjang & pendek)
    r'selamat pagi|selamat siang|selamat sore|selamat malam|'
    r'pagi|siang|sore|malam|'
    # Sapaan umum & informal (termasuk huruf berulang seperti halloo, haloo)
    r'hal+o+|hel+o+|hi|permisi|assalamu\'?alaikum|assalamualaikum|'
    # Sapaan daerah / Sunda
    r'punteun|punten|nuhun|'
    # Frasa permohonan bantuan
    r'mohon dibantu|mohon bantuan(?:nya)?|minta tolong|tolong dibantu|tolong bantuan(?:nya)?|'
    # Ucapan terima kasih dan penutup
    r'terima\s?kasih\s?sebelumnya|terimakasih\s?sebelumnya|'
    r'terima\s?kasih\s?banyak|terimakasih\s?banyak|'
    r'terima\s?kasih|terimakasih|makasih|'
    r'thanks|thank\s?you|guys'
    r')\b'
)
df['MASKED TICKET'] = df['MASKED TICKET'].apply(lambda x: re.sub(sapaan_pattern, 'sapaan', x, flags=re.IGNORECASE))

df.head()

,RAW TICKET,REPLACED NEWLINE,LOWERCASE,CLEANED TICKET,MASKED TICKET
0,Klien: Setwapres Medsos\nisu: Crawlback Commen...,Klien: Setwapres Medsos. isu: Crawlback Commen...,klien: setwapres medsos. isu: crawlback commen...,crawlback comment. siang tim it minta tolong b...,crawlback comment. sapaan tim it sapaan bantua...
1,"selamat pagi tim it, mohon bantuannya saya men...","selamat pagi tim it, mohon bantuannya saya men...","selamat pagi tim it, mohon bantuannya saya men...","selamat pagi tim it, mohon bantuannya saya men...","sapaan tim it, sapaan saya menemukan link dari..."
2,Klien: BPS\nIsu: Data postingan Instagram tida...,Klien: BPS. Isu: Data postingan Instagram tida...,klien: bps. isu: data postingan instagram tida...,data postingan instagram tidak masuk. selamat ...,data postingan instagram tidak masuk. sapaan t...
3,selamat sore mas @Dhanysybn dan tim info untuk...,selamat sore mas @Dhanysybn dan tim info untuk...,selamat sore mas @dhanysybn dan tim info untuk...,selamat sore mas @dhanysybn dan tim info untuk...,sapaan mas nama orang dan tim info untuk siput...
4,Klien: Heinz \nIsu: Dashboard loading\n\nSelam...,Klien: Heinz . Isu: Dashboard loading. Selamat...,klien: heinz . isu: dashboard loading. selamat...,"dashboard loading. selamat sore tim it, mohon ...","dashboard loading. sapaan tim it, sapaan ada k..."


#### Remove Emoticon

In [188]:
# Hapus emotikon
def hapus_emotikon(teks):
    if not isinstance(teks, str):
        return teks
    # Rentang emotikon umum (emoji, simbol, dll.)
    emotikon_pattern = r'[\U0001F600-\U0001F64F\U0001F300-\U0001F5FF\U0001F680-\U0001F6FF\U0001F700-\U0001F77F\U0001F780-\U0001F7FF\U0001F800-\U0001F8FF\U0001F900-\U0001F9FF\U0001FA00-\U0001FA6F\U0001FA70-\U0001FAFF]+'
    return re.sub(emotikon_pattern, '', teks)

df['REMOVED EMOJIS'] = df['MASKED TICKET'].apply(hapus_emotikon)
df.head()

,RAW TICKET,REPLACED NEWLINE,LOWERCASE,CLEANED TICKET,MASKED TICKET,REMOVED EMOJIS
0,Klien: Setwapres Medsos\nisu: Crawlback Commen...,Klien: Setwapres Medsos. isu: Crawlback Commen...,klien: setwapres medsos. isu: crawlback commen...,crawlback comment. siang tim it minta tolong b...,crawlback comment. sapaan tim it sapaan bantua...,crawlback comment. sapaan tim it sapaan bantua...
1,"selamat pagi tim it, mohon bantuannya saya men...","selamat pagi tim it, mohon bantuannya saya men...","selamat pagi tim it, mohon bantuannya saya men...","selamat pagi tim it, mohon bantuannya saya men...","sapaan tim it, sapaan saya menemukan link dari...","sapaan tim it, sapaan saya menemukan link dari..."
2,Klien: BPS\nIsu: Data postingan Instagram tida...,Klien: BPS. Isu: Data postingan Instagram tida...,klien: bps. isu: data postingan instagram tida...,data postingan instagram tidak masuk. selamat ...,data postingan instagram tidak masuk. sapaan t...,data postingan instagram tidak masuk. sapaan t...
3,selamat sore mas @Dhanysybn dan tim info untuk...,selamat sore mas @Dhanysybn dan tim info untuk...,selamat sore mas @dhanysybn dan tim info untuk...,selamat sore mas @dhanysybn dan tim info untuk...,sapaan mas nama orang dan tim info untuk siput...,sapaan mas nama orang dan tim info untuk siput...
4,Klien: Heinz \nIsu: Dashboard loading\n\nSelam...,Klien: Heinz . Isu: Dashboard loading. Selamat...,klien: heinz . isu: dashboard loading. selamat...,"dashboard loading. selamat sore tim it, mohon ...","dashboard loading. sapaan tim it, sapaan ada k...","dashboard loading. sapaan tim it, sapaan ada k..."


#### Parse Url

In [189]:
def ubah_url_ke_domain_revisi(teks):
    if not isinstance(teks, str):
        return teks
    
    # POLA REGEX BARU: 
    # Menangkap seluruh string URL yang valid (termasuk huruf, angka, strip, titik, dan garis miring)
    pola_url = r'https?://[\w\-\.\/\?\&\=\%]+'
    
    def ekstrak_domain(match):
        url = match.group(0)
        
        # Mencegah titik atau koma di akhir kalimat ikut terbaca sebagai bagian URL
        if url.endswith('.') or url.endswith(','):
            url = url[:-1]
            
        try:
            # urlparse akan membedah URL. 
            # Contoh: dari "https://megapolitan.kompas.com/..." kita ambil "megapolitan.kompas.com"
            netloc = urlparse(url).netloc
            
            # Pecah berdasarkan titik
            parts = netloc.split('.')
            
            # LOGIKA PENCARIAN NAMA DOMAIN UTAMA:
            # Kasus 1: Domain Indonesia 3 tingkat (contoh: news.detik.co.id -> ambil 'detik')
            if len(parts) >= 3 and parts[-2] in ['co', 'go', 'ac', 'or', 'sch', 'my']:
                domain_utama = parts[-3]
                
            # Kasus 2: Domain standar dengan/tanpa subdomain (contoh: megapolitan.kompas.com -> ambil 'kompas')
            elif len(parts) >= 2:
                domain_utama = parts[-2]
                
            # Kasus 3: Fallback jika format URL tidak biasa
            else:
                domain_utama = parts[0]
                
            # Jika domain utama tertangkap sebagai 'www', ambil kata setelahnya
            if domain_utama == 'www' and len(parts) >= 2:
                domain_utama = parts[-1]
                
            return f"tautan {domain_utama.lower()}"
            
        except Exception:
            return "tautan" # Fallback jika terjadi error parsing
            
    # Terapkan re.sub menggunakan fungsi ekstrak_domain
    teks = re.sub(pola_url, ekstrak_domain, teks)
    
    return teks

# Terapkan fungsi ke kolom yang sudah diproses sebelumnya
df['FINAL TICKET'] = df['REMOVED EMOJIS'].apply(ubah_url_ke_domain_revisi)
df.head()

,RAW TICKET,REPLACED NEWLINE,LOWERCASE,CLEANED TICKET,MASKED TICKET,REMOVED EMOJIS,FINAL TICKET
0,Klien: Setwapres Medsos\nisu: Crawlback Commen...,Klien: Setwapres Medsos. isu: Crawlback Commen...,klien: setwapres medsos. isu: crawlback commen...,crawlback comment. siang tim it minta tolong b...,crawlback comment. sapaan tim it sapaan bantua...,crawlback comment. sapaan tim it sapaan bantua...,crawlback comment. sapaan tim it sapaan bantua...
1,"selamat pagi tim it, mohon bantuannya saya men...","selamat pagi tim it, mohon bantuannya saya men...","selamat pagi tim it, mohon bantuannya saya men...","selamat pagi tim it, mohon bantuannya saya men...","sapaan tim it, sapaan saya menemukan link dari...","sapaan tim it, sapaan saya menemukan link dari...","sapaan tim it, sapaan saya menemukan link dari..."
2,Klien: BPS\nIsu: Data postingan Instagram tida...,Klien: BPS. Isu: Data postingan Instagram tida...,klien: bps. isu: data postingan instagram tida...,data postingan instagram tidak masuk. selamat ...,data postingan instagram tidak masuk. sapaan t...,data postingan instagram tidak masuk. sapaan t...,data postingan instagram tidak masuk. sapaan t...
3,selamat sore mas @Dhanysybn dan tim info untuk...,selamat sore mas @Dhanysybn dan tim info untuk...,selamat sore mas @dhanysybn dan tim info untuk...,selamat sore mas @dhanysybn dan tim info untuk...,sapaan mas nama orang dan tim info untuk siput...,sapaan mas nama orang dan tim info untuk siput...,sapaan mas nama orang dan tim info untuk siput...
4,Klien: Heinz \nIsu: Dashboard loading\n\nSelam...,Klien: Heinz . Isu: Dashboard loading. Selamat...,klien: heinz . isu: dashboard loading. selamat...,"dashboard loading. selamat sore tim it, mohon ...","dashboard loading. sapaan tim it, sapaan ada k...","dashboard loading. sapaan tim it, sapaan ada k...","dashboard loading. sapaan tim it, sapaan ada k..."


#### Save Results to CSV

In [190]:
# Simpan hasil akhir ke CSV baru
output_path = 'datasets/cleaned_tickets.csv'

df_result = df[['RAW TICKET', 'FINAL TICKET']].copy()
df_result.rename(columns={'FINAL TICKET': 'CLEANED TICKET'}, inplace=True)

# tampilkan tiket yang menjadi kosong setelah preprocessing dan sebelum yang menjadi kosong setelah preprocessing
print("Tiket yang menjadi kosong setelah preprocessing:")
print(df_result[df_result['CLEANED TICKET'].str.strip() == '']['RAW TICKET'].tolist())

df_result['panjang_teks'] = df_result['CLEANED TICKET'].str.split().str.len()

n_sebelum = len(df_result)
df_result = df_result[df_result['panjang_teks'] > 0].reset_index(drop=True)
n_sesudah = len(df_result)

print(f"Tiket dihapus (teks kosong setelah preprocessing): {n_sebelum - n_sesudah}")
print(f"Sisa data untuk clustering: {n_sesudah}")
df_result.to_csv(output_path, index=False)

# simpan hasil akhirnya saja ke txt
with open('datasets/cleaned_tickets.txt', 'w', encoding='utf-8') as f:
    for ticket in df_result['CLEANED TICKET']:
        f.write(ticket + '\n')

Tiket yang menjadi kosong setelah preprocessing:
[]
Tiket dihapus (teks kosong setelah preprocessing): 0
Sisa data untuk clustering: 1639


### Sentence-Embedding

#### Implements Sliding Windows

In [191]:
tokenizer = model.tokenizer

def get_sliding_window_embedding(teks, model, max_length=256, stride=128):
    original_max_length = tokenizer.model_max_length
    tokenizer.model_max_length = 100_000
    tokens = tokenizer.encode(teks, add_special_tokens=False)
    tokenizer.model_max_length = original_max_length

    # Jika pendek, langsung encode
    if len(tokens) <= max_length - 2:
        return model.encode(teks)

    window_size = max_length - 2  # ruang untuk [CLS] dan [SEP]
    step = window_size - stride    # seberapa jauh window bergeser tiap iterasi

    chunk_embeddings = []
    start = 0

    while start < len(tokens):
        end = min(start + window_size, len(tokens))
        chunk_tokens = tokens[start:end]
        chunk_text = tokenizer.decode(chunk_tokens)
        chunk_emb = model.encode(chunk_text)
        chunk_embeddings.append(chunk_emb)
        
        if end == len(tokens):  # sudah sampai akhir
            break
        start += step

    return np.mean(chunk_embeddings, axis=0)

#### Embed Sentence

In [192]:
# Mengaktifkan ekstensi pandas dari tqdm untuk memunculkan progress bar
tqdm.pandas(desc="Proses Embedding IndoSBERT")

# 1. Mencegah Error: Pastikan tidak ada data kosong (NaN) akibat proses pembersihan sebelumnya
# Mengubah NaN atau nilai kosong menjadi string kosong
df_result['CLEANED TICKET'] = df_result['CLEANED TICKET'].fillna("").astype(str)

# 2. Ekstraksi Embedding dengan Sliding Window
# Kita gunakan .progress_apply() sebagai pengganti .apply() biasa agar progress bar muncul
df_result['EMBEDDING'] = df_result['CLEANED TICKET'].progress_apply(
    lambda teks: get_sliding_window_embedding(
        teks=teks, 
        model=model, 
        max_length=256, 
        stride=128
    )
)

# Menampilkan 5 baris pertama untuk memastikan kolom EMBEDDING sudah terbentuk
df_result.head()

Proses Embedding IndoSBERT: 100%|██████████| 1639/1639 [10:58<00:00,  2.49it/s]


,RAW TICKET,CLEANED TICKET,panjang_teks,EMBEDDING
0,Klien: Setwapres Medsos\nisu: Crawlback Commen...,crawlback comment. sapaan tim it sapaan bantua...,43,"[0.000837568, -0.58585614, 0.06469053, 0.08374..."
1,"selamat pagi tim it, mohon bantuannya saya men...","sapaan tim it, sapaan saya menemukan link dari...",82,"[0.02750898, 0.037827887, -0.52839667, -0.1313..."
2,Klien: BPS\nIsu: Data postingan Instagram tida...,data postingan instagram tidak masuk. sapaan t...,34,"[-0.17697816, -0.045605347, -0.34718037, 0.424..."
3,selamat sore mas @Dhanysybn dan tim info untuk...,sapaan mas nama orang dan tim info untuk siput...,21,"[0.08603825, -0.017807066, 0.016980363, 0.3204..."
4,Klien: Heinz \nIsu: Dashboard loading\n\nSelam...,"dashboard loading. sapaan tim it, sapaan ada k...",17,"[0.14301422, 0.02800764, 0.030197056, 0.243817..."


In [ ]:
tqdm.pandas(desc="Proses Embedding IndoSBERT")

# embedding tanpa sliding window
df_result['EMBEDDING'] = df_result['CLEANED TICKET'].progress_apply(lambda teks: model.encode(teks))

df_result.head()

#### Save to Pickle

In [195]:
output_path = 'datasets/tickets.pkl'

df_result.to_pickle(output_path)

### Preprocessed Data for Word Cloud

In [196]:
keywords_to_keep = {
    'belum', 'tidak', 'kurang', 'bisa', 'semua', 'atas', 'bawah', 
    'masuk', 'keluar', 'hilang', 'mati', 'kosong', 'gagal', 'salah'
}

khas_tiket_noise = {
    'mas', 'mba', 'tim', 'it', 'mohon', 'dibantu', 'punteun', 'tolong', 
    'cek', 'bantu', 'halo', 'min', 'admin', 'dari', 'ke', 'perihal',
    'siang', 'pagi', 'sore', 'malam', 'selamat', 'terima', 'kasih', 'sebelumnya',
    'ya', 'yaa', 'kah', 'lah', 'pun', 'terimakasih', 'salam', 'sapaan', 'nama', 'orang'
}

factory = StopWordRemoverFactory()
stopword_indonesia = factory.get_stop_words()

stopwords = set(stopword_indonesia) - keywords_to_keep
stopwords.update(khas_tiket_noise)

In [197]:
df_wc = df_result[['CLEANED TICKET']].copy()
df_wc.rename(columns={'CLEANED TICKET': 'RAW TICKET'}, inplace=True)

df_wc['CLEANED TICKET'] = df_wc['RAW TICKET'].apply(lambda x: ' '.join([word for word in x.split() if word.lower() not in stopwords]))

df_wc.head()

,RAW TICKET,CLEANED TICKET
0,crawlback comment. sapaan tim it sapaan bantua...,crawlback comment. bantuan nya crawlback comme...
1,"sapaan tim it, sapaan saya menemukan link dari...","it, menemukan link megapolitan.kompas.com tida..."
2,data postingan instagram tidak masuk. sapaan t...,"data postingan instagram tidak masuk. it, dice..."
3,sapaan mas nama orang dan tim info untuk siput...,"info siputri tidak bisa akses mas, minta yaaa."
4,"dashboard loading. sapaan tim it, sapaan ada k...","dashboard loading. it, keluhan pihak heinz das..."


In [198]:
# Simpan hasil pembersihan untuk word cloud ke txt
with open('datasets/cleaned_tickets_for_wordcloud.txt', 'w', encoding='utf-8') as f:
    for ticket in df_wc['CLEANED TICKET']:
        f.write(ticket + '\n')

In [199]:
df_result['panjang_teks'] = df_result['CLEANED TICKET'].str.split().str.len()
print(df_result['panjang_teks'].value_counts().sort_index().head(20))

# Lihat tiket dengan teks sangat pendek
tiket_pendek = df_result[df_result['panjang_teks'] <= 5][['CLEANED TICKET', 'panjang_teks']]
print(f"\nJumlah tiket dengan ≤5 kata: {len(tiket_pendek)}")
print(tiket_pendek.head(20))


panjang_teks
2      1
3      1
4      1
5      7
6      5
7      8
8      3
9      8
10     4
11    10
12    12
13    16
14    18
15    26
16    38
17    42
18    55
19    45
20    66
21    69
Name: count, dtype: int64

Jumlah tiket dengan ≤5 kata: 10
                                     CLEANED TICKET  panjang_teks
562               kebutuhan report jam 7.30. sapaan             5
1125                              crawlback pending             2
1598          dibuatkan scraper media internasional             4
1620  permintaan pengambilan data instagram comment             5
1622           hasil screenshot blank tautan bisnis             5
1623                          halaman keyword error             3
1629                     data twitter hanya masuk 3             5
1630           hasil screenshot blank tautan bisnis             5
1632         kendala di crawlback facebook monalisa             5
1633               tidak masuk nna tautan tvonenews             5
